In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import scanpy as sp             
import scipy.sparse as sparse
import anndata as ad

In [2]:
adata = ad.read_h5ad("../data_interim/TMS_checked.h5ad")
print(adata.obs.columns)
unique_tissues = adata.obs['tissue'].unique()
for tissue in unique_tissues:
    safe_tissue_name = re.sub(r'[\\/*?:"<>|\s]', "_", str(tissue))
    
    adata_tissue = adata[adata.obs['tissue'] == tissue].copy()
    
    out_path = f'../data_interim/{safe_tissue_name}_checked.h5ad'
    
    adata_tissue.write(out_path)
    print(f"已保存: {out_path} (n_obs={adata_tissue.n_obs}, n_vars={adata_tissue.n_vars})")

Index(['age', 'cell', 'cell_ontology_class', 'cell_ontology_id',
       'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue',
       'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden'],
      dtype='object')
已保存: ../data_interim/spleen_marrow_checked.h5ad (n_obs=75938, n_vars=20138)
已保存: ../data_interim/Fat_checked.h5ad (n_obs=6777, n_vars=20138)
已保存: ../data_interim/Liver_checked.h5ad (n_obs=7294, n_vars=20138)
已保存: ../data_interim/Lung_checked.h5ad (n_obs=24540, n_vars=20138)
已保存: ../data_interim/Kidney_checked.h5ad (n_obs=21647, n_vars=20138)
已保存: ../data_interim/Heart_and_Aorta_checked.h5ad (n_obs=8613, n_vars=20138)


In [7]:
input_dir = Path('../data_interim')
output_dir = Path('../data_interim')
output_dir.mkdir(parents=True, exist_ok=True)

# 字段转换日志
conversion_log = []

def add_log(dataset, operation, old_field, new_field, status):
    conversion_log.append({
        "dataset": dataset,
        "operation": operation,
        "old_field": old_field,
        "new_field": new_field,
        "status": status
    })

def standardize_adata(adata, dataset_name):
    print(f"原始: {adata.n_obs} cells × {adata.n_vars} genes")
    print(list(adata.obs.columns))

    # 1. mouse_id_std
    if 'mouse.id' in adata.obs.columns:
        adata.obs['mouse_id_std'] = (
            adata.obs['mouse.id']
            .astype(str)
            .str.replace(r'[^a-zA-Z0-9\-]', '_', regex=True)
            .str.replace(r'_+', '_', regex=True)
            .str.strip('_')
            .astype('category')
        )
        add_log(dataset_name, "standardize", "mouse.id", "mouse_id_std", "success")

    # 2. tissue_std
    if 'tissue' in adata.obs.columns:
        adata.obs['tissue_std'] = (
            adata.obs['tissue']
            .astype(str)
            .str.replace(r'[^a-zA-Z0-9]', '_', regex=True)
            .str.replace(r'_+', '_', regex=True)
            .str.strip('_')
            .str.lower()
            .astype('category')
        )
        add_log(dataset_name, "standardize", "tissue", "tissue_std", "success")

    # 3. cell_type_original
    for col in ['cell_ontology_class', 'cell_type', 'celltype']:
        if col in adata.obs.columns:
            adata.obs['cell_type_original'] = adata.obs[col].astype('category')
            add_log(dataset_name, "standardize", col, "cell_type_original", "success")
            break

    # 4. age_months
    if 'age' in adata.obs.columns:
        adata.obs['age_months'] = (
            adata.obs['age']
            .astype(str)
            .str.extract(r'(\d+)')
            .astype(float)
        )
        add_log(dataset_name, "convert", "age", "age_months", "success")

    # 5. age_group
    if 'age_months' in adata.obs.columns:
        def age_group(x):
            # 历史标准化阶段标签；主分析以06冻结young=1/3、middle=18/21/24、old=30为准
            if pd.isna(x):
                return "unknown"
            elif x in (1, 3):
                return "young"
            elif x in (18, 21):
                return "middle"
            elif x in (24, 30):
                return "old"
            else:
                return "other"

        adata.obs['age_group'] = (
            adata.obs['age_months']
            .apply(age_group)
            .astype('category')
        )

    # 6. gene重复检查
    duplicate_genes = adata.var_names[adata.var_names.duplicated()]
    if len(duplicate_genes):
        add_log(dataset_name, "gene_check", "var_names", "duplicate_genes", f"{len(duplicate_genes)} duplicates")
        adata.var_names_make_unique()
    else:
        add_log(dataset_name, "gene_check", "var_names", "duplicate_genes", "none")

    # 7. Ensembl / gene symbol 检查
    sample = list(adata.var_names[:1000])
    ens_pattern = re.compile(r'^ENS(MUS)?G\d+')
    ens_ratio = sum(bool(ens_pattern.match(str(x))) for x in sample) / len(sample)

    if ens_ratio > 0.8:
        gene_type = "Ensembl_ID"
    elif ens_ratio < 0.2:
        gene_type = "Gene_symbol"
    else:
        gene_type = "Mixed"

    add_log(dataset_name, "gene_annotation", "var_names", gene_type, "checked")

    # 8. counts完整性
    if "counts" in adata.layers:
        X = adata.layers["counts"]
        count_source = "layers[counts]"
    else:
        X = adata.X
        count_source = "adata.X"

    if sparse.issparse(X):      
        values = X.data
    else:
        values = X.flatten()

    integer_counts = np.all(values.astype(int) == values)
    nan_check = np.isnan(values).any()

    add_log(
        dataset_name,
        "counts_check",
        count_source,
        "raw_counts",
        f"integer={integer_counts};nan={nan_check}"
    )

    return adata

# ================= 主流程 =================
h5ad_files = sorted(input_dir.glob("*_checked.h5ad"))

for file_path in h5ad_files:
    print("=" * 60)
    print(file_path.name)

    adata = sp.read_h5ad(file_path)    

    adata = standardize_adata(adata, file_path.stem)

    out_path = output_dir / (file_path.stem + "_standardized.h5ad")
    adata.write(out_path)
    print("保存:", out_path)

print("全部完成")

Fat_checked.h5ad
原始: 6777 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\Fat_checked_standardized.h5ad
Heart_and_Aorta_checked.h5ad
原始: 8613 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\Heart_and_Aorta_checked_standardized.h5ad
Kidney_checked.h5ad
原始: 21647 cells × 20138 genes
['age', 'cell', 'cell_ontology_class', 'cell_ontology_id', 'free_annotation', 'method', 'mouse.id', 'n_genes', 'sex', 'subtissue', 'tissue', 'tissue_free_annotation', 'n_counts', 'louvain', 'leiden']
保存: ..\data_interim\Kidney_checked_standardized.h5ad
Liver_checked.h5ad
原始: 7294 cells × 20138 genes
['age', 'cell', 'cell_ontology_cl

In [1]:
# 2026-11-03：逐组织交付审计由 scripts/12_remediate_analysis.py audit 生成。
import pandas as pd
for name in ["metadata_mapping", "gene_id_check", "counts_integrity_check", "standardization_report"]:
    report = pd.read_csv(f"../docs/{name}.tsv", sep="\t")
    assert report.dataset.nunique() == 7, f"{name} 未覆盖全部7个保留对象"
    print(name, report.shape, "7 standardized resources")


metadata_mapping (140, 7) 7 standardized resources
gene_id_check (140966, 6) 7 standardized resources
counts_integrity_check (7, 6) 7 standardized resources
standardization_report (7, 9) 7 standardized resources
